In [27]:
#Se tiver avisos importantes deixa aqui :D
# É pra ser só um notebook pra entregar, mas pra nao ficar mt grande por enquanto melhor deixar separadado (cuidado com os nomes de variaveis pra n repetir)

%pip install ucimlrepo
%pip install pandas
%pip install numpy
%pip install scikit-learn
%pip install matplotlib

In [28]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error,  root_mean_squared_error, r2_score)

In [29]:
#### Experimento B - Classificação de obesidade com base em hábitos alimentares e condição física ###
#Seleção e compreensão da base de dados

# dataset estimation_of_obesity_levels_based_on_eating_habits_and_physical_condition (obesidade para os mais próximos)
obesidade = fetch_ucirepo(id=544) 
df_obesidade = pd.concat([obesidade.data.features, obesidade.data.targets], axis=1)

linhas_obesidade = df_obesidade.shape[0]
colunas_obesidade = df_obesidade.shape[1]

#NObeyesdad: Classificação de obesidade (target)
classes_obesidade = df_obesidade['NObeyesdad'].unique()
contagem_classes_obesidade = df_obesidade['NObeyesdad'].value_counts()
qntd_classes_obesidade = len(classes_obesidade)
porcentagem_classes_obesidade = (contagem_classes_obesidade / linhas_obesidade) * 100

distribuicao_obesidade = pd.DataFrame({
    'Quantidade': contagem_classes_obesidade,
    'Porcentagem (%)': porcentagem_classes_obesidade.round(2)
})

print("Número de linhas:", linhas_obesidade)
print("Número de colunas:", colunas_obesidade)
print("Distribuição das classes:")
print(distribuicao_obesidade)

Número de linhas: 2111
Número de colunas: 17
Distribuição das classes:
                     Quantidade  Porcentagem (%)
NObeyesdad                                      
Obesity_Type_I              351            16.63
Obesity_Type_III            324            15.35
Obesity_Type_II             297            14.07
Overweight_Level_I          290            13.74
Overweight_Level_II         290            13.74
Normal_Weight               287            13.60
Insufficient_Weight         272            12.88


In [30]:
#Preparação dos dados

#spoiler: nao tem valores nulos
valores_nulos_obesidade = df_obesidade.isnull().sum()
#print("Valores nulos por coluna:", valores_nulos_obesidade)

#A variável alvo é Weight kkk
x_obesidade = df_obesidade.drop(['Weight', 'NObeyesdad'], axis=1)
y_obesidade = df_obesidade['Weight']

# Identificar colunas
numericas = x_obesidade.select_dtypes(include=['number']).columns
categoricas = x_obesidade.select_dtypes(include=['object', 'category']).columns

# MinMaxScaler (entre 0 e 1) pra normalizar os atributos numéricos e OneHotEncoder (0 ou 1) pra codificar os atributos categóricos
preprocessador = ColumnTransformer(
    transformers=[
        ('numericas', MinMaxScaler(), numericas),
        ('categoricas', OneHotEncoder(handle_unknown='ignore'), categoricas)
    ]
)

# Dividir os dados mantendo a proporção das classes (20% para teste e 80% para treino)
X_treino, X_teste, y_treino, y_teste = train_test_split(
    x_obesidade, y_obesidade,
    test_size=0.2,
    random_state=42,
)

# Treinar as transformações
X_treino = preprocessador.fit_transform(X_treino)
X_teste = preprocessador.transform(X_teste)

print("Treinamento:", X_treino.shape)
print("Teste:", X_teste.shape)
print("Estatísticas do peso:")
print(y_obesidade.describe())

print("Classes da variável-alvo:", y_obesidade.value_counts(), "\n")
print("\nQuantidade de atributos após processamento:", X_treino.shape[1], "\n")
print("\nQuantidade de registros de treinamento:", X_treino.shape[0], "\n")
print("\nQuantidade de registros de teste:", X_teste.shape[0], "\n")


Treinamento: (1688, 30)
Teste: (423, 30)
Estatísticas do peso:
count    2111.000000
mean       86.586058
std        26.191172
min        39.000000
25%        65.473343
50%        83.000000
75%       107.430682
max       173.000000
Name: Weight, dtype: float64
Classes da variável-alvo: Weight
80.000000    59
70.000000    43
50.000000    42
75.000000    40
60.000000    37
             ..
80.386078     1
68.850970     1
86.319887     1
88.037480     1
65.912688     1
Name: count, Length: 1525, dtype: int64 


Quantidade de atributos após processamento: 30 


Quantidade de registros de treinamento: 1688 


Quantidade de registros de teste: 423 



In [31]:
k = [5,10,15]

def k_neighbors_regressor(X_treino, y_treino, X_teste, y_teste, k):
    for i in k:
        knn = KNeighborsRegressor(n_neighbors=i)
        knn.fit(X_treino, y_treino)
        y_pred = knn.predict(X_teste)

        mae = mean_absolute_error(y_teste, y_pred)
        mse = mean_squared_error(y_teste, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_teste, y_pred)

        print(f"Resultados para k={i}:")
        print(f"MAE: {mae:.4f}")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"R²: {r2:.4f}\n")

In [32]:
def linear_regression(X_treino, y_treino, X_teste, y_teste):
    lr = LinearRegression()
    lr.fit(X_treino, y_treino)
    y_pred = lr.predict(X_teste)

    mae = mean_absolute_error(y_teste, y_pred)
    mse = mean_squared_error(y_teste, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_teste, y_pred)

    print("Resultados para Regressão Linear:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}\n")

In [33]:
def decision_tree_regressor(X_treino, y_treino, X_teste, y_teste):
    dt = DecisionTreeRegressor(random_state=42)
    dt.fit(X_treino, y_treino)
    y_pred = dt.predict(X_teste)

    mae = mean_absolute_error(y_teste, y_pred)
    mse = mean_squared_error(y_teste, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_teste, y_pred)

    print("Resultados para Árvore de Decisão:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²: {r2:.4f}\n")

In [34]:
k_neighbors_regressor(X_treino, y_treino, X_teste, y_teste, k)
linear_regression(X_treino, y_treino, X_teste, y_teste)
decision_tree_regressor(X_treino, y_treino, X_teste, y_teste)

Resultados para k=5:
MAE: 6.8201
MSE: 131.2490
RMSE: 11.4564
R²: 0.8139

Resultados para k=10:
MAE: 7.6018
MSE: 148.7114
RMSE: 12.1947
R²: 0.7891

Resultados para k=15:
MAE: 8.3731
MSE: 167.2864
RMSE: 12.9339
R²: 0.7628

Resultados para Regressão Linear:
MAE: 13.8194
MSE: 296.7836
RMSE: 17.2274
R²: 0.5791

Resultados para Árvore de Decisão:
MAE: 5.7625
MSE: 124.6776
RMSE: 11.1659
R²: 0.8232

